<a href="https://colab.research.google.com/github/DeepthiManthapuram/Building_LLM_Applications/blob/main/skill_map_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install langchain-tavily

In [6]:
pip install langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 34.3 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [19]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_tavily import  TavilySearch
from langchain.tools import tool
import requests
from langchain.agents import create_agent
from google.colab import userdata

load_dotenv()

google_api_key = userdata.get("GEMINI_API_KEY")

# Step 1: Initialize the Model

model = init_chat_model(
    model="gemini-3.5-flash",
    model_provider="google_genai",
    api_key=google_api_key
)

# Step 2: Create Skill Demand Search Tool (Tavily)
tavily_api_key = userdata.get("TAVILY_API_KEY")
skill_demand_tool = TavilySearch(
    max_results = 5,
    topic = "general",
    search_depth = "advanced",
    tavily_api_key = tavily_api_key
)

# Step 3: Create Custom Job Search Tool
rapidapi_key = userdata.get("RAPIDAPI_KEY")
@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring specific skill"""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    url = "https://jsearch.p.rapidapi.com/job-details"

    querystring = {"country":"in","job_id":"TU1SQkRfOURuWDhXVmFWREFBQUFBQT09OkVzd0JDb3dCUVVwcFZEUjBUR0pqWW5CcVdrOVFPVkpUWmpWSVdHNUJOelF4WXpRMlEwaGxPVFZwZUVSM2FsSkNlbEp6TFY5V1duWklabUo2U1hwSlQwZEtaR1U0UTNJd1p5MVdRM1JRWVdkYVMzWmFjR2s0ZEhScldXWTVSelJmV1VkVGVsYzJiMjlrY21aeGVUaGpWRGR0WVVkUkxXczJXSE0wTFdSc2VYbzRjWFJIWVRGUlZtUnFTMEppVEU4M1EwRVNGMGszZERaaGNsZFJUblF5UzNKMVJWQm5PSGsxYlZGUkdpSkJSSE55T1daUmNuWlRUbEpuUXpOYVpERlphbXMxWVVkSmNrSmZRMGN3VW05Qg"}

    headers = {
    "x-rapidapi-key": rapidapi_key,
    "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }

    response = requests.get(url, headers=headers, params=querystring)

    data = response.json()

    jobs = data.get("data", [])
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result

# Step 4: Define System Prompt
system_prompt = """
You are a Skill-to-Career Mapping assistant.

Your job is to help students understand:
1. Skill demand
2. Career trends
3. Required skills
4. Salary information
5. Related job opportunities

Available tools:

- skill_demand_tool:
  Use this to research skill demand, career trends, salary information,
  required skills, and industry information.

- search_jobs:
  Use this to find actual job opportunities related to the requested skill.

Instructions:

1. When the user asks about a skill, use skill_demand_tool to research
   the demand and career information.

2. Use search_jobs to find related job opportunities.

3. If the user specifies a location, search jobs in that location.

4. Do not invent job listings.

5. If the job search tool returns jobs, show them clearly.

6. If no jobs are found, clearly say that no matching jobs were found.

7. Give the final answer in a clean and readable format.

Use exactly this structure:

Skill Demand
Explain the current demand for the skill in 2-4 sentences.

Key Skills Required
- Skill 1
- Skill 2
- Skill 3
- Skill 4

Salary Information
- Fresher:
- Mid-level:
- Senior:

Related Jobs
1. Job Title
   Company:
   Location:
   Apply Link:

2. Job Title
   Company:
   Location:
   Apply Link:

Career Outlook
Give a short explanation of the career opportunities.

Important:
- Do not show tool calls.
- Do not show JSON.
- Do not show Python objects.
- Do not show internal reasoning.
- Do not mention Tavily, Gemini, or tool execution.
- Give only the final readable answer.
"""

# Step 5: Create and Run the Agent

agent = create_agent(
    model = model,
    tools = [skill_demand_tool, search_jobs],
    system_prompt = system_prompt,
    debug = True
)

user_query = "What is demand for data analyst?Provide the related jobs in India."
agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_query
            }
        ]
    }
)

[values] {'messages': [HumanMessage(content='What is demand for data analyst?Provide the related jobs in India.', additional_kwargs={}, response_metadata={}, id='128017a4-ad04-4a52-b6fb-077989601250')]}
[updates] {'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_jobs', 'arguments': '{"location": "India", "skill": "data analyst"}'}, '__gemini_function_call_thought_signatures__': {'call_569544': 'EpYKCpMKAWkUfRO307Q97ETMJcSPzH9JfnO9RPaBwHbwJ0dNrAIJZf8rQ1NA/mE1ojeVq8rMTA8vEzXLveso15v3K9fRpxFQxK/2f2bdW+ztKC5wu9wFGBjJTOKTiBAxsAsSFwocc0RMO0RDqjgqxBX4wes3Mi868UXEU9I5u/8o7PTir4hYhxxZXE++xRHg1DrMzeVF62QoYVK9HgGbDxYqz2ybA+5JycAdDBKx5ponSVHHwB8hsVL8e9DCuPxIHT4xnbVWZk3IcrZaLB9dzDEbf5XRuJGwYkvxfq+WezbxJOQ0HPOnmbRnqZXQ6ad8DA5W4B+HKWHotLQ28Sz8vOqVA+aUpQvhBRV6a3uzAdCPU6aMD+4nTZikpFfZg7xAStSWsQnQlK4dhaRwsSyVIhVeJgu4R7XmDGdAFdougtltHydGHw88pi+b6S6mzAY7OdEMzrZ0jGZKJDfd7NR8USdkMXO+LZLj3y3wB37+6eoEoTIT84p4OyIe1SQXWlTrWIW3v9mQtKY0i1dvFX/NjNM64F8MTiWbBqGLsYfgz

{'messages': [HumanMessage(content='What is demand for data analyst?Provide the related jobs in India.', additional_kwargs={}, response_metadata={}, id='128017a4-ad04-4a52-b6fb-077989601250'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_jobs', 'arguments': '{"location": "India", "skill": "data analyst"}'}, '__gemini_function_call_thought_signatures__': {'call_569544': 'EpYKCpMKAWkUfRO307Q97ETMJcSPzH9JfnO9RPaBwHbwJ0dNrAIJZf8rQ1NA/mE1ojeVq8rMTA8vEzXLveso15v3K9fRpxFQxK/2f2bdW+ztKC5wu9wFGBjJTOKTiBAxsAsSFwocc0RMO0RDqjgqxBX4wes3Mi868UXEU9I5u/8o7PTir4hYhxxZXE++xRHg1DrMzeVF62QoYVK9HgGbDxYqz2ybA+5JycAdDBKx5ponSVHHwB8hsVL8e9DCuPxIHT4xnbVWZk3IcrZaLB9dzDEbf5XRuJGwYkvxfq+WezbxJOQ0HPOnmbRnqZXQ6ad8DA5W4B+HKWHotLQ28Sz8vOqVA+aUpQvhBRV6a3uzAdCPU6aMD+4nTZikpFfZg7xAStSWsQnQlK4dhaRwsSyVIhVeJgu4R7XmDGdAFdougtltHydGHw88pi+b6S6mzAY7OdEMzrZ0jGZKJDfd7NR8USdkMXO+LZLj3y3wB37+6eoEoTIT84p4OyIe1SQXWlTrWIW3v9mQtKY0i1dvFX/NjNM64F8MTiWbBqGLsYfgzayK7f9XeHXGEB3Z05LqVyvOrStPZQuj9xatn4nqeov

In [24]:
response1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": user_query
            }
        ]
    }
)
print(response1["messages"][-1].content[0].text)

[values] {'messages': [HumanMessage(content='What is demand for data analyst?Provide the related jobs in India.', additional_kwargs={}, response_metadata={}, id='12b5fb8e-00ff-49d0-bad0-09699d35f3e9')]}


GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 59.201781483s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '